# Prediction Entropy and Information Budget

Where in a sequence and at which layer is the model uncertain or confident?
Track entropy of the predicted distribution across layers, measure per-token
surprisal, and find where the model commits to its prediction.

This notebook covers the `irtk.prediction_entropy` module.

## Why This Matters

Prediction entropy at each layer measures how 'decided' the model is about its output. Tracking when entropy drops reveals commitment points — the layers where the model transitions from uncertainty to confidence. Early commitment on easy tokens vs. late commitment on hard ones reflects the model's internal difficulty assessment.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import prediction_entropy

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Layer Prediction Entropy

How does prediction certainty evolve through layers?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

entropies = prediction_entropy.layer_prediction_entropy(model, tokens)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(entropies, cmap='viridis', aspect='auto')
ax.set_yticks(range(entropies.shape[0]))
ax.set_yticklabels(['Embed'] + [f'Layer {i}' for i in range(model.cfg.n_layers)])
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_title('Prediction Entropy by Layer and Position')
plt.colorbar(im, ax=ax, label='Entropy (nats)')
plt.tight_layout()
plt.show()

## 2. Prediction Commit Depth

At which layer does the model first commit to its prediction?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
paris_id = tokenizer.encode(" Paris")[0]

result = prediction_entropy.prediction_commit_depth(
    model, tokens, target_token=paris_id, pos=-1, threshold=0.1
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(result['probability_trajectory'], 'bo-', label=f'P(" Paris")')
if result['commit_layer'] is not None:
    ax.axvline(result['commit_layer'], color='red', linestyle='--',
               label=f'Commit layer = {result["commit_layer"]}')
ax.axhline(0.1, color='gray', linestyle=':', alpha=0.5, label='Threshold')
ax.set_xlabel('Layer (0=embed, 1+=after layer)')
ax.set_ylabel('Probability')
ax.set_title('Prediction Commit Depth for " Paris"')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Commit layer: {result['commit_layer']}")
print(f"Final probability: {result['final_probability']:.4f}")

## 3. Per-Token Surprisal

Which tokens does the model find most surprising?

In [ ]:
tokens = model.to_tokens("The cat sat on the mat and then jumped over the fence")
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

result = prediction_entropy.per_token_surprisal(model, tokens)

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(result['surprisals']))
colors = ['red' if i == result['most_surprising_pos'] else 'steelblue' for i in x]
ax.bar(x, result['surprisals'], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(token_strs[1:], rotation=45, ha='right')  # surprisal is for next token
ax.set_ylabel('Surprisal (nats)')
ax.set_title('Per-Token Surprisal')
ax.axhline(result['mean_surprisal'], color='gray', linestyle='--',
           label=f'Mean = {result["mean_surprisal"]:.2f}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Most surprising: '{token_strs[result['most_surprising_pos'] + 1]}' "
      f"(surprisal = {result['surprisals'][result['most_surprising_pos']]:.2f})")

## 4. Entropy Reduction by Layer

Which layers do the most "decision-making work"?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")

result = prediction_entropy.entropy_reduction_by_layer(model, tokens, pos=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result['entropy_per_layer'], 'bo-')
axes[0].set_xlabel('Component (0=embed, 1+=after layer)')
axes[0].set_ylabel('Entropy (nats)')
axes[0].set_title('Entropy Trajectory')

colors = ['green' if d > 0 else 'red' for d in result['delta_entropy']]
axes[1].bar(range(len(result['delta_entropy'])), result['delta_entropy'], color=colors)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Entropy reduction (+ = more certain)')
axes[1].set_title('Entropy Reduction per Layer')
axes[1].axhline(0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Biggest entropy reducer: layer {result['biggest_reducer']}")
print(f"Total reduction: {result['total_reduction']:.2f} nats")

## 5. Compare Entropy Profiles

Where does the model first treat two prompts differently?

In [ ]:
tokens_a = model.to_tokens("The Eiffel Tower is located in")
tokens_b = model.to_tokens("The weather today is going to")

result = prediction_entropy.compare_entropy_profiles(model, tokens_a, tokens_b, pos=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(result['entropy_a'], 'bo-', label='Eiffel Tower prompt')
axes[0].plot(result['entropy_b'], 'rs-', label='Weather prompt')
if result['divergence_layer'] is not None:
    axes[0].axvline(result['divergence_layer'], color='gray', linestyle='--',
                    label=f'Diverge at {result["divergence_layer"]}')
axes[0].set_xlabel('Component (0=embed, 1+=after layer)')
axes[0].set_ylabel('Entropy (nats)')
axes[0].set_title('Entropy Profiles')
axes[0].legend()

axes[1].bar(range(len(result['absolute_diff'])), result['absolute_diff'], color='purple')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('|Entropy difference|')
axes[1].set_title('Profile Divergence')

plt.tight_layout()
plt.show()

print(f"Divergence layer: {result['divergence_layer']}")
print(f"Max difference at layer: {result['max_diff_layer']}")

## Summary

| Function | Purpose |
|----------|--------|
| `layer_prediction_entropy()` | Entropy at every layer and position |
| `prediction_commit_depth()` | Earliest layer model commits to prediction |
| `per_token_surprisal()` | Per-token information content |
| `entropy_reduction_by_layer()` | Which layers reduce uncertainty most |
| `compare_entropy_profiles()` | Contrastive entropy analysis |